# Pre-processing and Modeling Notebook

This notebook covers the full workflow for the capstone modeling phase:
1. Load and validate cleaned data
2. Build preprocessing features for time-series modeling
3. Define expanding-window evaluation (24 months) and held-out test boundaries
4. Run one-step-ahead expanding-window evaluation for Seasonal Naive
5. Run one-step-ahead expanding-window evaluation for ETS
6. Evaluate 6 SARIMA-family candidates with one-step-ahead expanding-window cross-validation
7. Select the best SARIMA candidate and compare it against ETS and Seasonal Naive
8. Refit the overall winner on all pre-test data and evaluate once on held-out test
9. Save outputs for reporting and reproducibility

In [ ]:
# Core libraries
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

In [ ]:
# Paths
PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "call_data_cleaned.csv"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH

## 1. Load and Validate Input Data

In [ ]:
df = pd.read_csv(DATA_PATH)

# Ensure monthly datetime index
if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
elif "month" in df.columns:
    df["month"] = pd.to_datetime(df["month"])
    df = df.set_index("month")
else:
    raise ValueError("Expected a date column named 'date' or 'month'.")

df = df.sort_index()
df.index = pd.DatetimeIndex(df.index, freq="MS")

target_col = "total_calls"
if target_col not in df.columns:
    raise ValueError(f"Target column '{target_col}' not found.")

print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
display(df.head())

In [ ]:
# Basic quality checks
missing_summary = df.isna().sum().sort_values(ascending=False)
duplicate_index = df.index.duplicated().sum()

print("Missing values by column:")
display(missing_summary.to_frame("missing_count"))
print(f"Duplicate timestamps: {duplicate_index}")

## 2. Pre-processing and Feature Engineering

In [ ]:
model_df = df.copy()

# Target transformations and structural-break indicator
model_df["log_total_calls"] = np.log1p(model_df[target_col])
model_df["covid_period"] = (model_df.index >= pd.Timestamp("2020-03-01")).astype(int)

# Cyclical month encoding for optional downstream models
month_num = model_df.index.month
model_df["month_sin"] = np.sin(2 * np.pi * month_num / 12)
model_df["month_cos"] = np.cos(2 * np.pi * month_num / 12)

display(model_df[[target_col, "log_total_calls", "covid_period", "month_sin", "month_cos"]].head())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(model_df.index, model_df[target_col], label="Total Calls", linewidth=2)
ax.axvline(pd.Timestamp("2020-03-01"), color="crimson", linestyle="--", label="COVID-19 Wave 1")
ax.set_title("Monthly Total Calls with Structural Break Marker")
ax.set_xlabel("Date")
ax.set_ylabel("Calls")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Expanding-Window Evaluation Setup

Model selection uses a fixed-origin expanding window across the evaluation period.
At each step, the training window grows by one month and each model produces a one-step-ahead forecast.
This yields 24 evaluation points, all in the post-COVID regime, with no leakage.

- **Evaluation window:** 2022-01 to 2023-12 (24 one-step-ahead forecasts)
- **Held-out test:** 2024-01 to 2025-02 (14 months — never touched during model selection)

In [ ]:
# Evaluation and test window boundaries
eval_start = pd.Timestamp("2022-01-01")
eval_end   = pd.Timestamp("2023-12-01")
test_start = pd.Timestamp("2024-01-01")

y = model_df[target_col].copy()

y_eval = y.loc[eval_start:eval_end]
y_test = y.loc[test_start:]

# Pre-eval training data (initial expanding window seed)
y_pre_eval = y.loc[:eval_start - pd.offsets.MonthBegin(1)]

print(f"Total series length : {len(y)} months")
print(f"Initial training    : {len(y_pre_eval)} months (up to {y_pre_eval.index.max().date()})")
print(f"Evaluation window   : {len(y_eval)} months ({eval_start.date()} to {eval_end.date()})")
print(f"Held-out test       : {len(y_test)} months ({test_start.date()} to {y_test.index.max().date()})")

assert len(y_pre_eval) >= 24, "Initial training window must be at least 24 months."

In [ ]:
def evaluate_forecast(y_true, y_pred, model_name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = (np.abs((y_true - y_pred) / np.maximum(y_true, 1e-9))).mean() * 100
    return pd.DataFrame([{"model": model_name,
                           "MAE": round(mae, 1),
                           "RMSE": round(rmse, 1),
                           "MAPE_%": round(mape, 2)}])

## 4. Expanding-Window Model Evaluation

Each model is re-estimated at every step using all data up to (but not including) the forecast date,
then produces a single one-step-ahead forecast. The loop runs 24 times, once per evaluation month.

For SARIMA, we evaluate six candidates: (0,1,1), (1,1,0), (1,1,1), each with and without seasonal terms at lag 12.

In [ ]:
# 4.1 Seasonal Naive — expanding window
naive_rows = []

for cutoff in y_eval.index:
    y_seen   = y.loc[:cutoff - pd.offsets.MonthBegin(1)]
    ref_date = cutoff - pd.DateOffset(months=12)
    pred     = y_seen.loc[ref_date] if ref_date in y_seen.index else y_seen.iloc[-1]
    naive_rows.append({"date": cutoff, "actual": y.loc[cutoff], "forecast": pred})

ew_naive = pd.DataFrame(naive_rows).set_index("date")
print("Seasonal Naive done.")

In [ ]:
# 4.2 ETS (Holt-Winters) — expanding window
ets_rows = []

for cutoff in y_eval.index:
    y_seen = y.loc[:cutoff - pd.offsets.MonthBegin(1)]
    fit = ExponentialSmoothing(
        y_seen,
        trend="add",
        damped_trend=True,
        seasonal="add",
        seasonal_periods=12,
        initialization_method="estimated"
    ).fit(optimized=True)
    pred = fit.forecast(1).iloc[0]
    ets_rows.append({"date": cutoff, "actual": y.loc[cutoff], "forecast": pred})

ew_ets = pd.DataFrame(ets_rows).set_index("date")
print("ETS done.")

In [ ]:
# 4.3 SARIMA grid — expanding window (log-transformed target)
sarima_candidates = [
    {"order": (0, 1, 1), "seasonal_order": (0, 0, 0, 12), "label": "ARIMA(0,1,1)"},
    {"order": (1, 1, 0), "seasonal_order": (0, 0, 0, 12), "label": "ARIMA(1,1,0)"},
    {"order": (1, 1, 1), "seasonal_order": (0, 0, 0, 12), "label": "ARIMA(1,1,1)"},
    {"order": (0, 1, 1), "seasonal_order": (1, 1, 1, 12), "label": "SARIMA(0,1,1)x(1,1,1,12)"},
    {"order": (1, 1, 0), "seasonal_order": (1, 1, 1, 12), "label": "SARIMA(1,1,0)x(1,1,1,12)"},
    {"order": (1, 1, 1), "seasonal_order": (1, 1, 1, 12), "label": "SARIMA(1,1,1)x(1,1,1,12)"}
]

ew_sarima_candidates = {}
sarima_metrics_rows = []

for candidate in sarima_candidates:
    label = candidate["label"]
    order = candidate["order"]
    seasonal_order = candidate["seasonal_order"]
    candidate_rows = []
    print(f"Evaluating {label} ...")

    for i, cutoff in enumerate(y_eval.index):
        y_seen = y.loc[:cutoff - pd.offsets.MonthBegin(1)]
        y_seen_log = np.log1p(y_seen)

        fit = SARIMAX(
            y_seen_log,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False)

        pred = np.expm1(fit.get_forecast(steps=1).predicted_mean.iloc[0])
        candidate_rows.append({"date": cutoff, "actual": y.loc[cutoff], "forecast": pred})

        if (i + 1) % 8 == 0:
            print(f"  {label}: {i+1}/{len(y_eval)} steps done")

    ew_df = pd.DataFrame(candidate_rows).set_index("date")
    ew_sarima_candidates[label] = ew_df

    sarima_metrics_rows.append(evaluate_forecast(ew_df["actual"], ew_df["forecast"], label))

sarima_metrics_df = pd.concat(sarima_metrics_rows, ignore_index=True).sort_values("RMSE")
display(sarima_metrics_df)

best_sarima_name = sarima_metrics_df.iloc[0]["model"]
best_sarima_eval_df = ew_sarima_candidates[best_sarima_name]
best_sarima_cfg = next(c for c in sarima_candidates if c["label"] == best_sarima_name)
best_sarima_order = best_sarima_cfg["order"]
best_sarima_seasonal = best_sarima_cfg["seasonal_order"]
print(f"Best SARIMA-family candidate: {best_sarima_name}")

In [ ]:
# Compare best SARIMA candidate vs ETS vs Seasonal Naive on the same evaluation window
metrics_val = []
for name, ew_df in [("SeasonalNaive(12)", ew_naive),
                    ("ETS_Additive", ew_ets),
                    (best_sarima_name, best_sarima_eval_df)]:
    metrics_val.append(evaluate_forecast(ew_df["actual"], ew_df["forecast"], name))

metrics_val_df = pd.concat(metrics_val, ignore_index=True).sort_values("RMSE")
display(metrics_val_df)

best_model_name = metrics_val_df.iloc[0]["model"]
print(f"\nBest overall model by RMSE: {best_model_name}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Top panel: one-step-ahead forecasts vs actual
ax = axes[0]
ax.plot(y_eval.index, ew_naive["actual"],    label="Actual",         linewidth=2, color="black")
ax.plot(y_eval.index, ew_naive["forecast"],  label="Seasonal Naive", linestyle="--")
ax.plot(y_eval.index, ew_ets["forecast"],    label="ETS",            linestyle="--")
ax.plot(y_eval.index, best_sarima_eval_df["forecast"], label=f"Best SARIMA: {best_sarima_name}", linestyle="--")
ax.set_title("One-Step-Ahead Forecasts vs Actual (Evaluation Window)")
ax.set_ylabel("Total Calls")
ax.legend()

# Bottom panel: absolute error over time
ax2 = axes[1]
ax2.plot(y_eval.index, (ew_naive["actual"]  - ew_naive["forecast"]).abs(),  label="Naive |error|",  linestyle="--")
ax2.plot(y_eval.index, (ew_ets["actual"]    - ew_ets["forecast"]).abs(),    label="ETS |error|",    linestyle="--")
ax2.plot(y_eval.index, (best_sarima_eval_df["actual"] - best_sarima_eval_df["forecast"]).abs(), label="Best SARIMA |error|", linestyle="--")
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_title("Absolute Forecast Error Over Time (Expanding Window)")
ax2.set_xlabel("Date")
ax2.set_ylabel("|Error| (calls)")
ax2.legend()

plt.tight_layout()
fig.savefig(FIGURES_DIR / "expanding_window_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Refit Best Model on All Pre-Test Data and Evaluate on Held-Out Test

Now that we have selected the best model using the expanding window,
we refit it on all data up to the end of the evaluation period (2016-04 to 2023-12)
and evaluate once on the held-out test set (2024-01 to 2025-02).

In [ ]:
y_train_val = y.loc[:eval_end]

if best_model_name == "SeasonalNaive(12)":
    history = y_train_val.copy()
    preds = []
    for dt in y_test.index:
        ref_date = dt - pd.DateOffset(months=12)
        pred = history.loc[ref_date] if ref_date in history.index else history.iloc[-1]
        preds.append(pred)
        history.loc[dt] = y_test.loc[dt]
    y_pred_test = pd.Series(preds, index=y_test.index, name="best_model_forecast")

elif best_model_name == "ETS_Additive":
    best_fit = ExponentialSmoothing(
        y_train_val,
        trend="add",
        damped_trend=True,
        seasonal="add",
        seasonal_periods=12,
        initialization_method="estimated"
    ).fit(optimized=True)
    y_pred_test = best_fit.forecast(len(y_test))
    y_pred_test.index = y_test.index

else:
    y_train_val_log = np.log1p(y_train_val)
    best_fit = SARIMAX(
        y_train_val_log,
        order=best_sarima_order,
        seasonal_order=best_sarima_seasonal,
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False)
    y_fc_log = best_fit.get_forecast(steps=len(y_test)).predicted_mean
    y_pred_test = pd.Series(np.expm1(y_fc_log.values), index=y_test.index, name="best_model_forecast")

test_metrics_df = evaluate_forecast(y_test, y_pred_test, f"{best_model_name}_TEST")
display(test_metrics_df)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(y_train_val.index, y_train_val.values, label="Train+Validation", alpha=0.75)
ax.plot(y_test.index, y_test.values, label="Test Actual", linewidth=2)
ax.plot(y_pred_test.index, y_pred_test.values, label=f"Test Forecast ({best_model_name})", linestyle="--", linewidth=2)
ax.set_title("Final Held-out Test Forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Total Calls")
ax.legend()
plt.tight_layout()
plot_path = FIGURES_DIR / "final_test_forecast.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved plot: {plot_path}")

## 6. Save Artifacts

In [ ]:
val_metrics_path = MODELS_DIR / "validation_metrics.csv"
test_metrics_path = MODELS_DIR / "test_metrics.csv"
forecast_path = MODELS_DIR / "test_forecasts.csv"

metrics_val_df.to_csv(val_metrics_path, index=False)
test_metrics_df.to_csv(test_metrics_path, index=False)
pd.DataFrame({"actual": y_test, "forecast": y_pred_test}).to_csv(forecast_path)

print("Saved artifacts:")
print(f"- {val_metrics_path}")
print(f"- {test_metrics_path}")
print(f"- {forecast_path}")

## 7. Notes for Capstone Write-up

- Report expanding-window metrics (MAE, RMSE, MAPE) to justify model selection.
- Report held-out test metrics as the single final generalization estimate.
- Include the rolling absolute error plot to show *when* models struggle (early post-COVID months).
- Discuss COVID-era regime shift as the primary source of residual error spikes.
- Include a short error analysis for high-demand months and policy implications.